# Humanise IPv4 and IPv4+Port into Short Alphanumeric Codes

These functions convert an IPv4 address (and optionally a port) into a **short, case‑sensitive alphanumeric string** (base‑62). The reverse functions decode the string back to the original values.

**Why?**  
- Give users a memorable, easy‑to‑type identifier instead of a raw IP address.  
- Useful for peer‑to‑peer connections, game servers, SSH shortcuts, or any situation where typing a numeric IP is cumbersome.  
- The IP+port variant uses the **minimum number of characters** possible for each pair.

In [ ]:
import ipaddress

# Base62 alphabet: 0-9, A-Z, a-z
_B62 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
_B62R = {c: i for i, c in enumerate(_B62)}

def _to_b62(n: int, pad: int = None) -> str:
    s = ""
    while n:
        n, r = divmod(n, 62)
        s = _B62[r] + s
    s = s or "0"
    if pad:
        s = s.rjust(pad, "0")
    return s

def _from_b62(s: str) -> int:
    v = 0
    for c in s:
        v = v * 62 + _B62R[c]
    return v

## 1. Encode / Decode IPv4 Alone

**`humanise_ip(ip)`**  
Takes a dotted‑decimal IPv4 string, returns a **6‑character** base‑62 string (always fixed length).  
Example: `"192.168.1.1" → "E4GZvW"`

In [ ]:
def humanise_ip(ip: str) -> str:
    n = int(ipaddress.IPv4Address(ip))
    return _to_b62(n, pad=6)

def dehumanise_ip(code: str) -> str:
    return str(ipaddress.IPv4Address(_from_b62(code)))

## 2. Encode / Decode IPv4 + Port (Minimum Length)

**`humanise_ip_port(ip, port)`**  
Combines a 32‑bit IP and a 16‑bit port into a 48‑bit integer, then encodes it into **the shortest possible** base‑62 string (no padding).  
Length varies from 1 to 9 characters.  
Example: `("8.8.8.8", 53) → "1v1Gt6"`

In [ ]:
def humanise_ip_port(ip: str, port: int) -> str:
    n = (int(ipaddress.IPv4Address(ip)) << 16) | port
    return _to_b62(n)

def dehumanise_ip_port(code: str) -> tuple[str, int]:
    n = _from_b62(code)
    return str(ipaddress.IPv4Address(n >> 16)), n & 0xFFFF

## 3. Quick Test

Run the cell below to verify that encoding + decoding returns the original values.

In [ ]:
# Test IPv4 alone
ip1 = "192.168.1.100"
c1 = humanise_ip(ip1)
print(f"{ip1} -> {c1} -> {dehumanise_ip(c1)}")

# Test IPv4 + port
ip2, port2 = "8.8.8.8", 53
c2 = humanise_ip_port(ip2, port2)
dec_ip, dec_port = dehumanise_ip_port(c2)
print(f"{ip2}:{port2} -> {c2} -> {dec_ip}:{dec_port}")